In [15]:
# Binarization -> Generate a boundary and Mark them 0 and 1 (Important in image processing -> colour to black and white)

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import Binarizer

In [16]:
df = pd.read_csv('../../../DataBox/titanic_prepsd.csv')[['Age','Fare','SibSp','Parch','Survived']]
df.dropna(inplace=True)

In [17]:
# 2 column to one -> later we apply binarization on family column

df['family'] = df['SibSp'] + df['Parch']
df.drop(columns=['SibSp','Parch'],inplace=True)

# I/O
X = df.drop(columns=['Survived'])
y = df['Survived']

# TTS
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

# Without binarization

clf = DecisionTreeClassifier()

clf.fit(X_train,y_train)

y_pred = clf.predict(X_test)

print(accuracy_score(y_test,y_pred))
print(np.mean(cross_val_score(DecisionTreeClassifier(),X,y,cv=10,scoring='accuracy')))

0.6083916083916084
0.6429773082942096


In [18]:
# Applying Binarization

# 0 -> Not with family (travling Alone) // 1 -> travling with family
trf = ColumnTransformer([
    ('bin',Binarizer(copy=False),['family'])
],remainder='passthrough') # copy true -> 2 new column , false -> changes in existing column + 1

X_train_trf = trf.fit_transform(X_train)
X_test_trf = trf.transform(X_test)

print(pd.DataFrame(X_train_trf,columns=['family','Age','Fare']))

clf = DecisionTreeClassifier()
clf.fit(X_train_trf,y_train)
y_pred2 = clf.predict(X_test_trf)

print(accuracy_score(y_test,y_pred2))

X_trf = trf.fit_transform(X)
print(np.mean(cross_val_score(DecisionTreeClassifier(),X_trf,y,cv=10,scoring='accuracy')))

     family   Age      Fare
0       1.0  31.0   20.5250
1       1.0  26.0   14.4542
2       1.0  30.0   16.1000
3       0.0  33.0    7.7750
4       0.0  25.0   13.0000
..      ...   ...       ...
566     1.0  46.0   61.1750
567     0.0  25.0   13.0000
568     0.0  41.0  134.5000
569     1.0  33.0   20.5250
570     0.0  33.0    7.8958

[571 rows x 3 columns]
0.6153846153846154
0.6289710485133021
